# MMGTFFF — Corrected Preprocessing Pipeline (v3, single notebook)
### Multi-Modal Graph Transformer for Federated Financial Forecasting

Run top-to-bottom in a fresh Google Colab runtime. Self-contained: clones StockNet, pulls SEC EDGAR
fundamentals, and produces one final modeling parquet/csv.

**This notebook merges and fixes `MMGTFFF_stocknet_preprocessing.ipynb` and
`MMGTFFF_SECEDGAR_preprocessing.ipynb`.** Fixes applied vs. those two notebooks:

1. **Sector mapping** — derived directly from the repo's own `StockTable` file (all 88 tickers,
   authoritative), instead of a hand-typed dict that mixed in ~50 tickers that aren't even in our
   dataset (CLF, FCX, TSLA, IBM, ...) and required a silent fallback to `'Unknown'`.
2. **SEC EDGAR ticker list** — built from the *actual* 87 StockNet tickers (read from the cloned
   repo), instead of a hand-typed 134-ticker list that didn't match our tickers 1:1. The old list
   wasted API calls on 47 tickers we don't use (AIG, GS, IBM, TSLA, ...) and silently failed to look
   up real StockNet tickers whose CIK needed special-casing (e.g. `PCLN`).
3. **`TotalLiabilities` tag bug** — the old fallback tag `LiabilitiesAndStockholdersEquity` is
   *Liabilities + Equity*, not Liabilities. Using it as a stand-in for `TotalLiabilities` silently
   corrupts `Debt_To_Equity` for every company that lacks the primary `Liabilities` tag. Fixed by
   dropping that tag and instead backfilling via the accounting identity
   `TotalLiabilities = TotalAssets - StockholdersEquity` only when both are present.
4. **Missing fundamentals are no longer bare NaN** — per the roadmap ("do NOT zero-fill missing
   fundamentals without a missingness indicator"), every fundamental column now ships with a
   `*_missing` binary flag and a `Days_Since_Filing` counter, and is never zero-filled.
5. **Fundamental changes, not just levels** — adds quarter-over-quarter growth/delta features
   (Revenue growth, EPS growth, ROA change, margin change, etc.), forward-filled the same way as
   the levels.
6. **Tweet → trading-day alignment now respects market close** — the old pipeline bucketed every
   tweet into the *calendar date* embedded in its StockNet filename, including tweets posted after
   4:00pm ET, which is look-ahead leakage for next-day-close prediction (a tweet at 6pm Monday
   should not inform a same-day Monday close prediction — it should roll to the next trading day).
   This notebook parses each tweet's real `created_at` timestamp, converts to US/Eastern, and rolls
   any post-close (or weekend/holiday) tweet forward to the next actual trading day for that ticker.
7. Everything else (technical indicators computed on the full un-trimmed price history before any
   date trimming, StockNet-style target thresholds, company/event tweet split, point-in-time
   forward-fill of fundamentals by *filed* date not *period-end* date) is carried over unchanged —
   those parts of the original notebooks were already correct.

**Output:** `final/stocknet_final_modeling_set.parquet` (+ `.csv`), one row per (ticker, trading day).

## 1. Setup & Clone

In [ ]:
!git clone https://github.com/yumoxu/stocknet-dataset.git
print('Cloned!')

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import re
import time
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

PRICE_RAW = 'stocknet-dataset/price/raw/'
TWEET_PREPROCESSED = 'stocknet-dataset/tweet/preprocessed/'

price_tickers = sorted([f.replace('.csv', '') for f in os.listdir(PRICE_RAW) if f.endswith('.csv')])
tweet_tickers = sorted([d for d in os.listdir(TWEET_PREPROCESSED) if os.path.isdir(os.path.join(TWEET_PREPROCESSED, d))])

print(f'Price tickers: {len(price_tickers)}')
print(f'Tweet tickers: {len(tweet_tickers)}')

## 2. Sector Mapping — from the repo's own `StockTable`

FIX: derive the mapping straight from the source-of-truth file instead of a hand-typed dict, so
it can't silently mismatch our actual ticker set.

In [ ]:
with open('stocknet-dataset/StockTable', 'r') as f:
    lines = [l.rstrip('\n') for l in f.readlines()]

header = lines[0].split('\t')
SECTOR_MAP = {}
for line in lines[1:]:
    if not line.strip():
        continue
    parts = line.split('\t')
    sector_raw, symbol_raw = parts[0], parts[1]
    ticker = symbol_raw.lstrip('$').strip()
    sector = sector_raw.strip().replace(' ', '_').replace('Matierials', 'Materials')  # fix repo typo
    SECTOR_MAP[ticker] = sector

def get_sector(ticker):
    return SECTOR_MAP.get(ticker, 'Unknown')

unmapped = [t for t in price_tickers if t not in SECTOR_MAP]
print(f'Sectors found: {sorted(set(SECTOR_MAP.values()))}')
print(f'Mapped: {len(SECTOR_MAP)} tickers | Unmapped from our price set: {unmapped if unmapped else "NONE"}')

## 3. Process Raw Prices → Returns, Technical Indicators, Target

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line

def process_price_file(filepath, ticker):
    df = pd.read_csv(filepath)
    if 'Date' not in df.columns:
        df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume']
    else:
        df = df.rename(columns={'Adj Close': 'Adj_Close'})

    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    # Guard: raw StockNet price files are already one row per trading day with no gaps
    # filled in — consecutive rows ARE consecutive trading days. This is what later
    # sliding-window code relies on.
    assert df['Date'].is_monotonic_increasing and not df['Date'].duplicated().any(), \
        f'{ticker}: price rows are not a clean monotonic trading-day sequence'

    price_col = 'Adj_Close' if 'Adj_Close' in df.columns else 'Close'

    df['Return'] = df[price_col].pct_change()
    df['Movement_Pct'] = df['Return'] * 100

    # StockNet/ACL18 target convention: >=0.55% up, <=-0.5% down, buffer zone dropped
    conditions = [df['Movement_Pct'] > 0.55, df['Movement_Pct'] < -0.5]
    df['Target'] = np.select(conditions, [1, 0], default=np.nan)

    df['MA_5'] = df[price_col].rolling(5).mean()
    df['MA_10'] = df[price_col].rolling(10).mean()
    df['MA_20'] = df[price_col].rolling(20).mean()
    df['Price_MA5_Ratio'] = df[price_col] / df['MA_5']
    df['Price_MA10_Ratio'] = df[price_col] / df['MA_10']
    df['Price_MA20_Ratio'] = df[price_col] / df['MA_20']
    df['Volatility_5'] = df['Return'].rolling(5).std()
    df['Volatility_20'] = df['Return'].rolling(20).std()
    df['RSI_14'] = compute_rsi(df[price_col], 14)
    df['MACD'], df['MACD_Signal'] = compute_macd(df[price_col])
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']
    df['Volume_Change'] = df['Volume'].pct_change()
    df['HL_Spread'] = (df['High'] - df['Low']) / df['Close']

    df['Ticker'] = ticker
    df['Sector'] = get_sector(ticker)
    # Per-ticker trading-day index — lets any downstream windowing assert consecutiveness
    df['Trading_Day_Index'] = np.arange(len(df))
    return df

all_price_dfs, price_errors = [], []
for ticker in tqdm(price_tickers, desc='Processing prices'):
    try:
        all_price_dfs.append(process_price_file(os.path.join(PRICE_RAW, f'{ticker}.csv'), ticker))
    except Exception as e:
        price_errors.append((ticker, str(e)))

price_df = pd.concat(all_price_dfs, ignore_index=True)
print(f'\nPrice data: {price_df.shape}')
print(f'Tickers: {price_df["Ticker"].nunique()}')
print(f'Date range: {price_df["Date"].min()} to {price_df["Date"].max()}')
print(f'Errors: {price_errors if price_errors else "None"}')

# Per-ticker sorted trading-day calendars — used below for the tweet close-cutoff rollover
TRADING_CALENDARS = {
    t: np.sort(price_df.loc[price_df['Ticker'] == t, 'Date'].values) for t in price_tickers
}

## 4. Process Tweets — Company vs. Event Split, WITH Market-Close Cutoff

**FIX (leakage):** the original notebook bucketed every tweet by the calendar date baked into the
StockNet filename. A tweet posted at 9pm ET is still labeled with that day's date, so it would leak
into "today's" feature row even though it arrived after the market had already closed. We now parse
each tweet's real `created_at` timestamp, convert to US/Eastern, and roll any tweet posted at/after
16:00 ET (or on a non-trading day) forward to the ticker's *next* actual trading day.

In [ ]:
ALL_TICKERS = set(price_tickers)
TICKER_PATTERNS = {f'${t.lower()}' for t in ALL_TICKERS} | {f'${t}' for t in ALL_TICKERS}

MACRO_KEYWORDS = [
    'fed ', 'fomc', 'interest rate', 'rate hike', 'rate cut', 'taper',
    'quantitative easing', 'yellen', 'bernanke', 'central bank',
    'gdp', 'jobs report', 'unemployment', 'nonfarm', 'payroll',
    'inflation', 'cpi', 'consumer price', 'housing starts', 'retail sales',
    'sp500', 's&p 500', 's&p500', 'dow jones', 'nasdaq', 'russell 2000',
    'vix', 'the market', 'wall street', 'stock market', 'bull market',
    'bear market', 'market crash', 'correction', 'rally', 'selloff',
    'sell-off', 'all time high', 'circuit breaker',
    'oil price', 'crude oil', 'opec', 'gold price', 'commodity',
    'earnings season', 'sector rotation', 'tech sector', 'financials',
    'energy sector', 'healthcare sector',
    'brexit', 'sanctions', 'trade war',
    'stimulus', 'shutdown', 'debt ceiling', 'fiscal cliff',
    'geopolitical', 'regulation', 'legislation'
]

def classify_tweet(text, current_ticker):
    text_lower = text.lower()
    current_patterns = {f'${current_ticker.lower()}', f'${current_ticker}'}
    other_mentions = sum(1 for p in TICKER_PATTERNS if p in text_lower and p not in current_patterns)
    if other_mentions >= 2:
        return 'event'
    if any(kw in text_lower for kw in MACRO_KEYWORDS):
        return 'event'
    return 'company'

MARKET_CLOSE_HOUR = 16  # 4:00 PM US/Eastern
EASTERN = ZoneInfo('America/New_York')
UTC = ZoneInfo('UTC')

def parse_tweet_time(created_at):
    # StockNet format: 'Mon Jan 06 18:39:16 +0000 2014'
    dt_utc = datetime.strptime(created_at, '%a %b %d %H:%M:%S %z %Y')
    return dt_utc.astimezone(EASTERN)

def trading_day_for(ticker, calendar_date, eastern_dt):
    """Roll a tweet forward to the next actual trading day if it landed after market
    close, or on a day the ticker didn't trade at all (weekend/holiday)."""
    calendar = TRADING_CALENDARS[ticker]
    target = np.datetime64(calendar_date)

    if eastern_dt.hour >= MARKET_CLOSE_HOUR or eastern_dt.hour == MARKET_CLOSE_HOUR and eastern_dt.minute > 0:
        # Posted at/after close -> earliest trading day strictly after this calendar date
        idx = np.searchsorted(calendar, target, side='right')
    else:
        # Posted before close -> this day if it's a trading day, else the next one
        idx = np.searchsorted(calendar, target, side='left')

    if idx >= len(calendar):
        return None  # no future trading day in our window (near the end of the dataset)
    return pd.Timestamp(calendar[idx])

def process_tweets_split(ticker):
    tweet_path = os.path.join(TWEET_PREPROCESSED, ticker)
    if not os.path.exists(tweet_path):
        return pd.DataFrame()

    by_day = {}  # trading_day -> {'company': [...], 'event': [...]}
    for date_file in sorted(os.listdir(tweet_path)):
        filepath = os.path.join(tweet_path, date_file)
        try:
            calendar_date = datetime.strptime(date_file, '%Y-%m-%d').date()
        except ValueError:
            continue

        try:
            with open(filepath, 'r') as f:
                tweets = [json.loads(line) for line in f if line.strip()]
        except Exception:
            continue

        for t in tweets:
            if not isinstance(t, dict):
                continue
            text = t.get('text', '')
            if isinstance(text, list):
                text = ' '.join(text)
            text = text.strip()
            if not text:
                continue

            created_at = t.get('created_at')
            if created_at:
                try:
                    eastern_dt = parse_tweet_time(created_at)
                except ValueError:
                    eastern_dt = datetime(calendar_date.year, calendar_date.month, calendar_date.day,
                                           12, 0, tzinfo=EASTERN)
            else:
                eastern_dt = datetime(calendar_date.year, calendar_date.month, calendar_date.day,
                                       12, 0, tzinfo=EASTERN)

            trading_day = trading_day_for(ticker, calendar_date, eastern_dt)
            if trading_day is None:
                continue

            label = classify_tweet(text, ticker)
            bucket = by_day.setdefault(trading_day, {'company': [], 'event': []})
            bucket[label].append(text)

    records = []
    for trading_day, texts in by_day.items():
        company_texts, event_texts = texts['company'], texts['event']
        if len(company_texts) + len(event_texts) == 0:
            continue
        records.append({
            'Ticker': ticker,
            'Date': trading_day,
            'Company_Tweet_Count': len(company_texts),
            'Company_Texts': ' [SEP] '.join(company_texts),
            'Event_Tweet_Count': len(event_texts),
            'Event_Texts': ' [SEP] '.join(event_texts),
            'Total_Tweet_Count': len(company_texts) + len(event_texts),
        })
    return pd.DataFrame(records)

all_tweet_dfs = []
for ticker in tqdm(tweet_tickers, desc='Splitting tweets (close-cutoff aligned)'):
    df = process_tweets_split(ticker)
    if len(df) > 0:
        all_tweet_dfs.append(df)

tweet_split_df = pd.concat(all_tweet_dfs, ignore_index=True)

total_comp = tweet_split_df['Company_Tweet_Count'].sum()
total_evt = tweet_split_df['Event_Tweet_Count'].sum()
total_all = total_comp + total_evt
print(f'Total tweet-day rows: {len(tweet_split_df)}')
print(f'Tickers with tweets: {tweet_split_df["Ticker"].nunique()}')
print(f'Total tweets: {total_all:,} | Company: {total_comp:,} ({100*total_comp/total_all:.1f}%) | Event: {total_evt:,} ({100*total_evt/total_all:.1f}%)')

## 5. Merge Price + Tweets, Trim to Tweet Window, Build Classification Target

In [ ]:
merged_df = price_df.merge(
    tweet_split_df[['Ticker', 'Date', 'Company_Tweet_Count', 'Company_Texts',
                     'Event_Tweet_Count', 'Event_Texts', 'Total_Tweet_Count']],
    on=['Ticker', 'Date'], how='left'
)
for col in ['Company_Tweet_Count', 'Event_Tweet_Count', 'Total_Tweet_Count']:
    merged_df[col] = merged_df[col].fillna(0).astype(int)
for col in ['Company_Texts', 'Event_Texts']:
    merged_df[col] = merged_df[col].fillna('')

print(f'Merged shape: {merged_df.shape}')

# StockNet tweet coverage window
merged_df = merged_df[(merged_df['Date'] >= '2014-01-01') & (merged_df['Date'] <= '2015-12-31')].copy()
print(f'After date trim: {merged_df.shape}')

# Drop warm-up rows (rolling indicators need 20+ days of prior history)
merged_df = merged_df.dropna(subset=['Return', 'MA_20', 'RSI_14']).copy()
print(f'After dropping warm-up NaN: {merged_df.shape}')
print(f'Unknown sectors: {(merged_df["Sector"] == "Unknown").sum()}')

clf_df = merged_df.dropna(subset=['Target']).copy()
clf_df['Target'] = clf_df['Target'].astype(int)
clf_df = clf_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f'\nClassification dataset: {clf_df.shape}')
print(clf_df['Target'].value_counts())
print(f'Balance: {clf_df["Target"].mean():.3f}')

FINAL_TICKERS = sorted(clf_df['Ticker'].unique())
print(f'Final ticker set: {len(FINAL_TICKERS)} tickers')

## 6. SEC EDGAR — Ticker → CIK Mapping

**FIX:** use `FINAL_TICKERS` (the 87 tickers that actually made it into the dataset above) instead
of a hand-typed 134-ticker list. Every ticker we look up here is a ticker we will actually use.

In [ ]:
HEADERS = {
    # SEC requires a descriptive User-Agent with a real contact email — replace before running.
    'User-Agent': 'MMGTFFF-Research your_email@example.com',
    'Accept-Encoding': 'gzip, deflate'
}

resp = requests.get('https://www.sec.gov/files/company_tickers.json', headers=HEADERS)
ticker_data = resp.json()

ticker_to_cik = {}
for _, val in ticker_data.items():
    ticker_to_cik[val['ticker'].upper()] = str(val['cik_str']).zfill(10)

print(f'Total tickers in SEC mapping: {len(ticker_to_cik)}')

# Known StockNet-era ticker -> current/alternate SEC ticker overrides
TICKER_OVERRIDES = {
    'GOOG': 'GOOGL',   # Alphabet Class C -> Class A CIK (same company)
    'FB': 'META',      # Facebook -> Meta rebrand
    'PCLN': 'BKNG',    # Priceline Group -> Booking Holdings rebrand
}

matched, unmatched = {}, []
for t in FINAL_TICKERS:
    lookup = t.replace('-', '.')
    if t in ticker_to_cik:
        matched[t] = ticker_to_cik[t]
    elif lookup in ticker_to_cik:
        matched[t] = ticker_to_cik[lookup]
    elif t in TICKER_OVERRIDES and TICKER_OVERRIDES[t] in ticker_to_cik:
        matched[t] = ticker_to_cik[TICKER_OVERRIDES[t]]
    else:
        unmatched.append(t)

print(f'Matched: {len(matched)} | Unmatched: {len(unmatched)}')
if unmatched:
    print(f'Unmatched (expected — mostly foreign private issuers that do not file US GAAP 10-K/10-Q): {sorted(unmatched)}')

## 7. Fetch Fundamentals from EDGAR XBRL

**FIX:** dropped `LiabilitiesAndStockholdersEquity` from the `TotalLiabilities` tag list — that tag
is *Liabilities + Equity*, not Liabilities alone, and using it as a substitute silently corrupts
`Debt_To_Equity` downstream. We backfill `TotalLiabilities` via the accounting identity
(`Assets - StockholdersEquity`) later instead, only when both real values are present.

In [ ]:
METRIC_TAGS = {
    'Revenue': [
        'RevenueFromContractWithCustomerExcludingAssessedTax',
        'Revenues', 'SalesRevenueNet', 'SalesRevenueGoodsNet',
        'RevenueFromContractWithCustomerIncludingAssessedTax',
    ],
    'NetIncome': ['NetIncomeLoss', 'NetIncomeLossAvailableToCommonStockholdersBasic', 'ProfitLoss'],
    'TotalAssets': ['Assets'],
    'TotalLiabilities': ['Liabilities'],  # no more LiabilitiesAndStockholdersEquity fallback here
    'StockholdersEquity': ['StockholdersEquity', 'StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest'],
    'OperatingIncome': ['OperatingIncomeLoss'],
    'EPS': ['EarningsPerShareBasic', 'EarningsPerShareDiluted'],
    'Cash': ['CashAndCashEquivalentsAtCarryingValue', 'CashCashEquivalentsAndShortTermInvestments'],
}

def get_company_facts(cik):
    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
    resp = requests.get(url, headers=HEADERS)
    return resp.json() if resp.status_code == 200 else None

def extract_metric(facts_data, tag_variants, start_date='2013-01-01', end_date='2016-06-30'):
    if not facts_data or 'facts' not in facts_data:
        return []
    us_gaap = facts_data['facts'].get('us-gaap', {})
    for tag in tag_variants:
        if tag not in us_gaap:
            continue
        units = us_gaap[tag].get('units', {})
        values = units.get('USD', units.get('USD/shares', []))
        if not values:
            continue
        results = []
        for entry in values:
            filed, end, form, val = entry.get('filed', ''), entry.get('end', ''), entry.get('form', ''), entry.get('val')
            if form not in ['10-K', '10-Q', '10-K/A', '10-Q/A', '20-F']:
                continue
            if filed < start_date or filed > end_date:
                continue
            if val is not None:
                results.append({'filed': filed, 'period_end': end, 'value': val, 'form': form})
        if results:
            return results
    return []

all_fundamentals, failed_tickers, no_data_tickers = [], [], []
for ticker, cik in tqdm(matched.items(), desc='Fetching from EDGAR'):
    try:
        facts = get_company_facts(cik)
        if facts is None:
            failed_tickers.append(ticker)
            time.sleep(0.15)
            continue
        ticker_has_data = False
        for metric_name, tag_variants in METRIC_TAGS.items():
            for entry in extract_metric(facts, tag_variants):
                all_fundamentals.append({
                    'Ticker': ticker, 'Metric': metric_name, 'Filed_Date': entry['filed'],
                    'Period_End': entry['period_end'], 'Value': entry['value'], 'Form': entry['form']
                })
                ticker_has_data = True
        if not ticker_has_data:
            no_data_tickers.append(ticker)
        time.sleep(0.15)
    except Exception as e:
        failed_tickers.append((ticker, str(e)))
        time.sleep(0.15)

fund_df = pd.DataFrame(all_fundamentals)
fund_df['Filed_Date'] = pd.to_datetime(fund_df['Filed_Date'])
fund_df['Period_End'] = pd.to_datetime(fund_df['Period_End'])

print(f'Total records: {len(fund_df)} | Tickers with data: {fund_df["Ticker"].nunique()}')
print(f'Failed API calls: {failed_tickers if failed_tickers else "None"}')
print(f'No data found (expected — foreign filers, non-GAAP tags): {no_data_tickers if no_data_tickers else "None"}')

## 8. Point-in-Time Alignment: Forward-Fill by *Filed* Date, With Missingness Flags,
### Days-Since-Filing, and Quarter-over-Quarter Change Features

This is the core roadmap fix for fundamentals:
- Forward-fill from the **filing date** (not period-end) — a filing can only affect predictions
  made *after* it became public.
- Every level column ships with a `_Missing` flag (1 = genuinely unavailable, not zero).
- `Days_Since_Filing` tells the model how stale the fundamental snapshot is.
- Growth/change columns capture *changes* in fundamentals, not just raw levels, per the roadmap.

In [ ]:
FUNDAMENTAL_COLS = ['Revenue', 'NetIncome', 'TotalAssets', 'TotalLiabilities',
                     'StockholdersEquity', 'OperatingIncome', 'EPS', 'Cash']

RATIO_DEFS = {
    'Profit_Margin': ('NetIncome', 'Revenue'),
    'Debt_To_Equity': ('TotalLiabilities', 'StockholdersEquity'),
    'ROA': ('NetIncome', 'TotalAssets'),
    'Current_Ratio': ('Cash', 'TotalLiabilities'),
    'Asset_Turnover': ('Revenue', 'TotalAssets'),
    'Operating_Margin': ('OperatingIncome', 'Revenue'),
}
ALL_FUNDAMENTAL_COLS = FUNDAMENTAL_COLS + list(RATIO_DEFS.keys())

def align_ticker_fundamentals(ticker, fund_raw, daily_dates):
    result = pd.DataFrame({'Date': daily_dates})
    daily_index = pd.DatetimeIndex(daily_dates)
    ticker_data = fund_raw[fund_raw['Ticker'] == ticker]

    if len(ticker_data) == 0:
        for col in ALL_FUNDAMENTAL_COLS:
            result[col] = np.nan
            result[f'{col}_Missing'] = 1
        result['Days_Since_Filing'] = np.nan
        result['Ticker'] = ticker
        return result

    last_filed_series = pd.Series(pd.NaT, index=daily_index)
    all_filed_dates = np.sort(ticker_data['Filed_Date'].unique())

    for metric in FUNDAMENTAL_COLS:
        metric_data = (ticker_data[ticker_data['Metric'] == metric]
                        .sort_values('Filed_Date')
                        .drop_duplicates(subset=['Filed_Date'], keep='last'))

        if len(metric_data) == 0:
            result[metric] = np.nan
            result[f'{metric}_Missing'] = 1
            result[f'{metric}_Growth'] = np.nan
            continue

        # Level, forward-filled by filing date
        ts = metric_data.set_index('Filed_Date')['Value']
        ts = ts[~ts.index.duplicated(keep='last')]
        ts_daily = ts.reindex(daily_index, method='ffill')
        result[metric] = ts_daily.values
        result[f'{metric}_Missing'] = result[metric].isna().astype(int)

        # Quarter-over-quarter growth at the filing level, then forward-filled the same way
        growth_at_filing = ts.pct_change().replace([np.inf, -np.inf], np.nan)
        growth_daily = growth_at_filing.reindex(daily_index, method='ffill')
        result[f'{metric}_Growth'] = growth_daily.values

    # Days since the most recent filing of ANY metric, as of each day
    idx_pos = np.searchsorted(all_filed_dates, daily_index.values, side='right') - 1
    days_since = np.full(len(daily_index), np.nan)
    valid = idx_pos >= 0
    days_since[valid] = (daily_index.values[valid] - all_filed_dates[idx_pos[valid]]) / np.timedelta64(1, 'D')
    result['Days_Since_Filing'] = days_since

    # Ratios computed from forward-filled levels (identity-consistent, no separate leakage risk)
    for ratio_name, (num, den) in RATIO_DEFS.items():
        result[ratio_name] = result[num] / result[den]
    result = result.replace([np.inf, -np.inf], np.nan)
    for ratio_name in RATIO_DEFS:
        result[f'{ratio_name}_Missing'] = result[ratio_name].isna().astype(int)

    # Backfill TotalLiabilities via accounting identity where the raw tag was missing
    # but Assets and Equity are both known (fixes the removed bad fallback tag).
    can_backfill = result['TotalLiabilities'].isna() & result['TotalAssets'].notna() & result['StockholdersEquity'].notna()
    result.loc[can_backfill, 'TotalLiabilities'] = (
        result.loc[can_backfill, 'TotalAssets'] - result.loc[can_backfill, 'StockholdersEquity']
    )
    result.loc[can_backfill, 'TotalLiabilities_Missing'] = 0

    result['Ticker'] = ticker
    return result

aligned_dfs = []
for ticker in tqdm(FINAL_TICKERS, desc='Aligning fundamentals (point-in-time)'):
    daily_dates = clf_df.loc[clf_df['Ticker'] == ticker, 'Date'].values
    aligned_dfs.append(align_ticker_fundamentals(ticker, fund_df, daily_dates))

fund_aligned = pd.concat(aligned_dfs, ignore_index=True)
print(f'Aligned shape: {fund_aligned.shape}')
print('\nCoverage (non-missing %):')
for col in ALL_FUNDAMENTAL_COLS:
    pct = 100 * (1 - fund_aligned[f'{col}_Missing'].mean())
    print(f'  {col}: {pct:.1f}%')

## 9. Merge Everything Into the Final Modeling Set

In [ ]:
final_df = clf_df.merge(fund_aligned, on=['Ticker', 'Date'], how='left')

# Any ticker with zero EDGAR coverage at all (e.g. foreign filers) still needs missing-flags set
missing_flag_cols = [f'{c}_Missing' for c in ALL_FUNDAMENTAL_COLS]
for col in missing_flag_cols:
    final_df[col] = final_df[col].fillna(1).astype(int)

print(f'Final shape: {final_df.shape}')
print(f'Tickers: {final_df["Ticker"].nunique()}')
print(f'Date range: {final_df["Date"].min()} to {final_df["Date"].max()}')
print(f'\nColumns ({len(final_df.columns)}):')
print(list(final_df.columns))

## 10. Save Outputs

In [ ]:
os.makedirs('final', exist_ok=True)

sector_df = pd.DataFrame([{'Ticker': t, 'Sector': get_sector(t)} for t in FINAL_TICKERS])
sector_df.to_csv('final/sector_mapping.csv', index=False)

fund_df.to_csv('final/edgar_raw_fundamentals.csv', index=False)
fund_aligned.to_parquet('final/edgar_daily_aligned.parquet', index=False)

final_df.to_csv('final/stocknet_final_modeling_set.csv', index=False)
final_df.to_parquet('final/stocknet_final_modeling_set.parquet', index=False)

print('Saved:')
!ls -lh final/

## 11. Final Validation

In [ ]:
print('=== FINAL VALIDATION ===')
v = pd.read_parquet('final/stocknet_final_modeling_set.parquet')
print(f'Shape: {v.shape}')
print(f'Tickers: {v["Ticker"].nunique()}')
print(f'Target balance: {v["Target"].mean():.3f}')
print(f'Rows with company text: {(v["Company_Texts"] != "").sum()} ({100*(v["Company_Texts"] != "").mean():.1f}%)')
print(f'Rows with event text: {(v["Event_Texts"] != "").sum()} ({100*(v["Event_Texts"] != "").mean():.1f}%)')

for col in ALL_FUNDAMENTAL_COLS:
    pct_present = 100 * (1 - v[f'{col}_Missing'].mean())
    print(f'{col}: {pct_present:.1f}% present (flagged, not zero-filled)')

# Sanity check: no ticker has duplicate (Ticker, Date) rows, and each ticker's dates are
# a monotonically increasing subsequence of its own trading calendar (no fabricated gaps).
dupes = v.duplicated(subset=['Ticker', 'Date']).sum()
print(f'\nDuplicate (Ticker, Date) rows: {dupes}')
for t in v['Ticker'].unique()[:5]:
    dates = v.loc[v['Ticker'] == t, 'Date']
    assert dates.is_monotonic_increasing, f'{t}: dates not sorted'
print('Spot-checked 5 tickers: dates monotonic within ticker.')